In [7]:
import os
import sys

from src.nba_scrapping import *
from src.utils import *
from src.config import *

from datetime import datetime

In [8]:
start_time = datetime.now()

# Retry errors for historical boxscores (safety)

In [9]:
#retry_seasons = ["2012-13"]

retry_seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(1999, 2025)]
#retry_seasons = ["2010-11"]


for retry_season in retry_seasons:
    print(f"--- Retry des boxscores échoués pour la saison {retry_season} ---")
    
    season_batch_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, retry_season)
    
    retry_failed_boxscores_for_season(season_batch_dir, max_retries=10)


--- Retry des boxscores échoués pour la saison 1999-00 ---
[INFO] Retrying 0 GAME_IDs for season folder data/raw/boxscores/batches/1999-00
✅ Retry process completed.
--- Retry des boxscores échoués pour la saison 2000-01 ---
[INFO] Retrying 0 GAME_IDs for season folder data/raw/boxscores/batches/2000-01
✅ Retry process completed.
--- Retry des boxscores échoués pour la saison 2001-02 ---
[INFO] Retrying 0 GAME_IDs for season folder data/raw/boxscores/batches/2001-02
✅ Retry process completed.
--- Retry des boxscores échoués pour la saison 2002-03 ---
[INFO] Retrying 0 GAME_IDs for season folder data/raw/boxscores/batches/2002-03
✅ Retry process completed.
--- Retry des boxscores échoués pour la saison 2003-04 ---
[INFO] Retrying 0 GAME_IDs for season folder data/raw/boxscores/batches/2003-04
✅ Retry process completed.
--- Retry des boxscores échoués pour la saison 2004-05 ---
[INFO] Retrying 0 GAME_IDs for season folder data/raw/boxscores/batches/2004-05
✅ Retry process completed.
--- 

# Saisons ciblées


In [13]:
seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2000, 2025)]
#seasons = ["2010-11"]

seasons


['2000-01',
 '2001-02',
 '2002-03',
 '2003-04',
 '2004-05',
 '2005-06',
 '2006-07',
 '2007-08',
 '2008-09',
 '2009-10',
 '2010-11',
 '2011-12',
 '2012-13',
 '2013-14',
 '2014-15',
 '2015-16',
 '2016-17',
 '2017-18',
 '2018-19',
 '2019-20',
 '2020-21',
 '2021-22',
 '2022-23',
 '2023-24',
 '2024-25']

# Merge batches and remove duplicate for all endpoints

In [14]:
for season in seasons:
    print(f"--- Merging boxscores endpoints for {season} ---")
    
    season_batch_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, season)
    
    merge_boxscore_batches_for_season(season_batch_dir)


--- Merging boxscores endpoints for 2000-01 ---
[MERGED] traditional saved to data/raw/boxscores/batches/2000-01/merged_batches/merged_traditional.csv (32555 rows)
[MERGED] advanced saved to data/raw/boxscores/batches/2000-01/merged_batches/merged_advanced.csv (30137 rows)
[MERGED] fourfactors saved to data/raw/boxscores/batches/2000-01/merged_batches/merged_fourfactors.csv (30137 rows)
[MERGED] misc saved to data/raw/boxscores/batches/2000-01/merged_batches/merged_misc.csv (30137 rows)
[MERGED] scoring saved to data/raw/boxscores/batches/2000-01/merged_batches/merged_scoring.csv (30137 rows)
[MERGED] usage saved to data/raw/boxscores/batches/2000-01/merged_batches/merged_usage.csv (30137 rows)
--- Merging boxscores endpoints for 2001-02 ---
[MERGED] traditional saved to data/raw/boxscores/batches/2001-02/merged_batches/merged_traditional.csv (32429 rows)
[MERGED] advanced saved to data/raw/boxscores/batches/2001-02/merged_batches/merged_advanced.csv (30125 rows)
[MERGED] fourfactors s

# Merge endpoints csv into one final with all columns


In [15]:
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

for season in seasons:
    print(f"--- Merging all endpoints csv into final for {season} ---")
    
    season_merged_batch_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, season, "merged_batches")
    
    all_boxscores_merged_filepath = merge_all_boxscore_stats(season_merged_batch_dir, output_filename=f"final_merged_all_boxscores_{run_timestamp}.csv")
    
    all_boxscores_merged_df = pd.read_csv(all_boxscores_merged_filepath, dtype={'gameId': str})
    
    #analyze_redundant_columns(all_boxscores_merged_df)
    

--- Merging all endpoints csv into final for 2000-01 ---
Shape before cleaning: (28808, 164)
[CLEAN] Renamed and dropped 55 redundant columns.
[CLEAN] Dropped 9 highly correlated columns.
Shape after cleaning: (28808, 100)
✅ All endpoints merged into: data/raw/boxscores/batches/2000-01/merged_batches/final_merged_all_boxscores_2025-11-01_03-02-18.csv (28808 rows)
--- Merging all endpoints csv into final for 2001-02 ---
Shape before cleaning: (30104, 164)
[CLEAN] Renamed and dropped 55 redundant columns.
[CLEAN] Dropped 9 highly correlated columns.
Shape after cleaning: (30104, 100)
✅ All endpoints merged into: data/raw/boxscores/batches/2001-02/merged_batches/final_merged_all_boxscores_2025-11-01_03-02-18.csv (30104 rows)
--- Merging all endpoints csv into final for 2002-03 ---
Shape before cleaning: (28976, 164)
[CLEAN] Renamed and dropped 55 redundant columns.
[CLEAN] Dropped 9 highly correlated columns.
Shape after cleaning: (28976, 100)
✅ All endpoints merged into: data/raw/boxscor

# 🧩 Merge all seasons final boxscores into one


In [16]:
print("\n[MERGE] Concatenating all final_merged_all_boxscores_*.csv into one DataFrame")
final_files = []

for season in seasons:
    season_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, season, "merged_batches")
    for file in os.listdir(season_dir):
        if file.startswith("final_merged_all_boxscores_") and file.endswith(".csv"):
            final_files.append(os.path.join(season_dir, file))

if not final_files:
    raise FileNotFoundError("❌ Aucun fichier final_merged_all_boxscores_*.csv trouvé.")

merged_df = pd.concat([pd.read_csv(f, dtype={'gameId': str}) for f in final_files], ignore_index=True)

os.makedirs(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR, exist_ok=True)
output_path = os.path.join(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR, f"all_seasons_boxscores_merged_{run_timestamp}.csv")
merged_df.to_csv(output_path, index=False)
print(f"✅ Sauvegardé dans : {output_path}")


[MERGE] Concatenating all final_merged_all_boxscores_*.csv into one DataFrame
✅ Sauvegardé dans : data/raw_last/batches_merged/all_seasons_boxscores_merged_2025-11-01_03-02-18.csv


# --- Merge games from each season (take most recent per season) ---


In [17]:
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

all_games = []

#use custom format because of the way we store the games by periods or not for scrapping
season_dirs = [dir for dir in os.listdir(DATA_GAMES_DIR) if os.path.isdir(os.path.join(DATA_GAMES_DIR, dir))]

print(season_dirs)

for season_dir in season_dirs:
    season_games_dir = os.path.join(DATA_GAMES_DIR, season_dir)
    if os.path.exists(season_games_dir):
        game_files = [os.path.join(season_games_dir, f) for f in os.listdir(season_games_dir) if f.endswith('.csv')]
        if game_files:
            latest_file = max(game_files, key=os.path.getmtime)
            df_games = pd.read_csv(latest_file, dtype={'GAME_ID': str})
            all_games.append(df_games)

if all_games:
    all_games_df = pd.concat(all_games, ignore_index=True)
    # games_output_path = os.path.join(DATA_LAST_GAMES_MERGED_DIR, f"games_merged_all_seasons_{run_timestamp}.csv")
    
    # os.makedirs(DATA_LAST_GAMES_MERGED_DIR, exist_ok=True)
    
    # all_games_df.to_csv(games_output_path, index=False)
    
    games_output_path = save_dataframe_to_csv(all_games_df, DATA_LAST_GAMES_MERGED_DIR, 'games_merged_all_seasons', run_timestamp)

    
    print(f"✅ Merged games saved to {games_output_path} ({len(all_games_df)} rows)")


['2010-11_2014-15', '2015-16_2019-20', '2000-01_2004-05', '2001-02_2001-02', '2005-06_2009-10', '2020-21_2024-25']
✅ Merged games saved to data/raw_last/games_merged/games_merged_all_seasons_2025-11-01_03-03-28.csv (66772 rows)


In [18]:
#store end time of notebook
end_time = datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-11-01 03:03:29.046710
Total time:  0:04:33.535269


In [19]:
print("\n✅ Scraping historique V3 terminé")



✅ Scraping historique V3 terminé
